# 🩺 Breast Cancer AI Advisor — Gradio UI
### Part 3 of 3 — Interactive Patient Report Interface

**Author:** Maryam Rafaqat

---
Upload any breast histopathology image → get CNN prediction percentages + full patient report.

**Requires files from Part 1 & 2:**
- `binary_model.h5`, `benign_subtype_model.h5`, `malignant_subtype_model.h5`
- `label_maps.json`
- Mistral-7B (downloaded from HuggingFace — needs GPU)

## 1. Install Dependencies

In [ ]:
!pip install -q --upgrade "scipy>=1.14.0"
!pip install -q gradio transformers accelerate bitsandbytes sentencepiece \
               tensorflow opencv-python-headless rouge-score bert-score
print('✅ All packages installed.')

## 2. Configuration — Set Your Models Directory

In [ ]:
import os, json, warnings
import cv2
import numpy as np
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.models import load_model
warnings.filterwarnings('ignore')

# ── UPDATE THIS to where your .h5 and .json files are ─────────────────────
MODELS_DIR = '/kaggle/input/breakhis-models'   # e.g. '/kaggle/input/breakhis-models'

IMG_SIZE = 224

BENIGN_SUBTYPES = {
    'adenosis': 0, 'fibroadenoma': 1,
    'phyllodes_tumor': 2, 'tubular_adenoma': 3,
}
MALIGNANT_SUBTYPES = {
    'ductal_carcinoma': 0, 'lobular_carcinoma': 1,
    'mucinous_carcinoma': 2, 'papillary_carcinoma': 3,
}
BENIGN_IDX_TO_NAME    = {v: k for k, v in BENIGN_SUBTYPES.items()}
MALIGNANT_IDX_TO_NAME = {v: k for k, v in MALIGNANT_SUBTYPES.items()}

def _p(f): return str(Path(MODELS_DIR) / f)

# Load label maps if available
if Path(_p('label_maps.json')).exists():
    with open(_p('label_maps.json')) as f:
        lm = json.load(f)
    BENIGN_SUBTYPES    = lm['benign_subtypes']
    MALIGNANT_SUBTYPES = lm['malignant_subtypes']
    BENIGN_IDX_TO_NAME    = {int(v): k for k, v in BENIGN_SUBTYPES.items()}
    MALIGNANT_IDX_TO_NAME = {int(v): k for k, v in MALIGNANT_SUBTYPES.items()}
    print('✓ label_maps.json loaded')

print('Loading CNN models...')
binary_model    = load_model(_p('binary_model.h5'))
benign_model    = load_model(_p('benign_subtype_model.h5'))
malignant_model = load_model(_p('malignant_subtype_model.h5'))
print('✓ All 3 CNN models loaded')

## 3. Load Mistral-7B

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID   = 'mistralai/Mistral-7B-Instruct-v0.2'
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)
print(f'Loading {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True,
)
print('✅ Mistral-7B ready.')

## 4. Core Pipeline Functions

In [ ]:
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — safe for Gradio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import io
from PIL import Image as PILImage

# ── CLAHE preprocessing ────────────────────────────────────────────────────
def apply_clahe(img_bgr):
    lab   = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def preprocess(img_path):
    img = cv2.imread(str(img_path))
    if img is None:
        raise ValueError(f'Cannot read image: {img_path}')
    img = apply_clahe(img)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img / 255.0

# ── CNN inference ──────────────────────────────────────────────────────────
def _uncertainty(probs):
    margin = np.max(probs) - np.sort(probs)[-2]
    return 'LOW' if margin >= 0.40 else ('MODERATE' if margin >= 0.20 else 'HIGH')

def run_cnn(img_path):
    batch = np.expand_dims(preprocess(img_path), 0)
    mal   = float(binary_model.predict(batch, verbose=0)[0][0])
    ben   = round(1 - mal, 4); mal = round(mal, 4)
    pred  = 'malignant' if mal > 0.5 else 'benign'
    conf  = mal if pred == 'malignant' else ben

    if pred == 'benign':
        raw = benign_model.predict(batch, verbose=0)[0]
        idx_map = BENIGN_IDX_TO_NAME
    else:
        raw = malignant_model.predict(batch, verbose=0)[0]
        idx_map = MALIGNANT_IDX_TO_NAME

    top  = int(np.argmax(raw))
    diff = {idx_map[i]: round(float(p), 4)
            for i, p in sorted(enumerate(raw), key=lambda x: -x[1])}
    return {
        'binary':        pred,
        'binary_conf':   round(conf, 4),
        'ben_prob':      ben,
        'mal_prob':      mal,
        'subtype':       idx_map[top],
        'subtype_conf':  round(float(raw[top]), 4),
        'uncertainty':   _uncertainty(raw),
        'differential':  diff,
        'display_label': f"{pred.capitalize()}: {idx_map[top].replace('_',' ').title()}",
    }

# ── Confidence chart ───────────────────────────────────────────────────────
def make_confidence_chart(result):
    labels = [k.replace('_',' ').title() for k in result['differential']]
    values = [v * 100 for v in result['differential'].values()]
    is_mal = result['binary'] == 'malignant'
    colors = ['#d32f2f' if is_mal else '#388e3c'] + ['#90a4ae'] * (len(labels) - 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4),
                             gridspec_kw={'width_ratios': [1, 2]})
    fig.patch.set_facecolor('#1a1a2e')

    # Left: binary donut
    ax0 = axes[0]
    ax0.set_facecolor('#1a1a2e')
    mal_p, ben_p = result['mal_prob'] * 100, result['ben_prob'] * 100
    donut_vals   = [mal_p, ben_p]
    donut_colors = ['#d32f2f', '#388e3c']
    wedges, _ = ax0.pie(
        donut_vals, colors=donut_colors,
        startangle=90, counterclock=False,
        wedgeprops=dict(width=0.45, edgecolor='#1a1a2e', linewidth=2),
    )
    centre_txt = f"{result['binary_conf']*100:.1f}%"
    ax0.text(0, 0.08, centre_txt, ha='center', va='center',
             fontsize=22, fontweight='bold', color='white')
    verdict = '🔴 MALIGNANT' if is_mal else '🟢 BENIGN'
    ax0.text(0, -0.22, verdict, ha='center', va='center',
             fontsize=12, fontweight='bold', color='white')
    ax0.text(0, -0.42, result['display_label'].split(': ')[1],
             ha='center', va='center', fontsize=9, color='#b0bec5')
    legend_patches = [
        mpatches.Patch(color='#d32f2f', label=f'Malignant {mal_p:.1f}%'),
        mpatches.Patch(color='#388e3c', label=f'Benign {ben_p:.1f}%'),
    ]
    ax0.legend(handles=legend_patches, loc='lower center',
               fontsize=8, facecolor='#1a1a2e', labelcolor='white',
               framealpha=0.4, bbox_to_anchor=(0.5, -0.15))
    ax0.set_title('Binary Classification', color='white', fontsize=11,
                  fontweight='bold', pad=8)

    # Right: subtype horizontal bars
    ax1 = axes[1]
    ax1.set_facecolor('#16213e')
    bars = ax1.barh(labels[::-1], values[::-1], color=colors[::-1],
                    edgecolor='#1a1a2e', height=0.5)
    for bar, val in zip(bars, values[::-1]):
        ax1.text(min(val + 1, 98), bar.get_y() + bar.get_height() / 2,
                 f'{val:.1f}%', va='center', ha='left',
                 color='white', fontsize=10, fontweight='bold')
    ax1.set_xlim(0, 105)
    ax1.set_xlabel('Probability (%)', color='#b0bec5', fontsize=9)
    ax1.set_title('Subtype Differential', color='white', fontsize=11, fontweight='bold')
    ax1.tick_params(colors='white', labelsize=9)
    ax1.spines[:].set_color('#37474f')
    ax1.set_facecolor('#16213e')
    unc_color = {'LOW': '#4caf50', 'MODERATE': '#ff9800', 'HIGH': '#f44336'}[result['uncertainty']]
    ax1.text(0.98, -0.12,
             f"AI Uncertainty: {result['uncertainty']}",
             transform=ax1.transAxes, ha='right', fontsize=9,
             color=unc_color, fontweight='bold')

    plt.tight_layout(pad=1.5)
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=130, bbox_inches='tight',
                facecolor='#1a1a2e')
    plt.close(fig)
    buf.seek(0)
    return PILImage.open(buf)

# ── LLM report generation ──────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a compassionate, expert AI cancer advisor. A patient has received \
an AI-assisted breast cancer classification from a histopathology image. Explain the result in \
clear, caring, informative language — like a knowledgeable friend who is also a medical expert.

Generate a detailed patient report with EXACTLY these 7 sections:

1. WHAT WE FOUND
   Explain the diagnosis (benign/malignant and the specific subtype) in simple plain language.

2. IS THIS DANGEROUS?
   Rate danger: LOW / MODERATE / HIGH. Include survival rates or prognosis.

3. DO YOU NEED TO SEE A DOCTOR?
   Give urgency: ROUTINE / SOON (weeks) / URGENT (days). Name the specialist.

4. WHAT TREATMENT MIGHT BE NEEDED?
   Cover surgery, chemo, radiation, hormone therapy, targeted therapy for this specific subtype.

5. IS CHEMOTHERAPY NECESSARY?
   Answer directly for this subtype. Explain what factors decide it.

6. WARNING SIGNS — WHEN TO SEEK IMMEDIATE HELP
   List specific red-flag symptoms for this cancer type.

7. IMPORTANT DISCLAIMER
   Remind the patient this is AI, not a replacement for a qualified doctor.

Rules: Use plain language. Be honest but compassionate. No unexplained jargon."""

def generate_report(result):
    diff_lines = '\n'.join(
        f"  {i+1}. {k.replace('_',' ').title():<28} {v*100:.1f}%"
        for i, (k, v) in enumerate(result['differential'].items())
    )
    unc_note = {
        'LOW'     : 'The AI is highly confident in this result.',
        'MODERATE': 'The AI has moderate confidence — pathologist review is especially important.',
        'HIGH'    : 'WARNING: AI confidence is LOW — professional pathologist review is essential.',
    }[result['uncertainty']]

    user_msg = (
        f"PATIENT AI CANCER SCREENING RESULT\n"
        f"{'─'*50}\n"
        f"  Classification : {result['binary'].upper()}\n"
        f"  Subtype        : {result['subtype'].replace('_',' ').title()}\n"
        f"  Binary confidence  : {result['binary_conf']*100:.1f}%  "
        f"(Malignant: {result['mal_prob']*100:.1f}%, Benign: {result['ben_prob']*100:.1f}%)\n"
        f"  Subtype confidence : {result['subtype_conf']*100:.1f}%\n"
        f"  AI uncertainty     : {result['uncertainty']}\n"
        f"  Note               : {unc_note}\n\n"
        f"Subtype probability breakdown:\n{diff_lines}\n\n"
        f"Please generate the full 7-section patient advisory report."
    )
    prompt = f"[INST] {SYSTEM_PROMPT}\n\n{user_msg} [/INST]"
    inputs = tokenizer(prompt, return_tensors='pt',
                       truncation=True, max_length=1536)
    inputs = {k: v.to(llm.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = llm.generate(
            **inputs, max_new_tokens=700,
            temperature=0.3, top_p=0.85,
            do_sample=True, repetition_penalty=1.15,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    n = inputs['input_ids'].shape[1]
    return tokenizer.decode(out[0][n:], skip_special_tokens=True).strip()

# ── ROUGE + BERTScore ──────────────────────────────────────────────────────
from rouge_score import rouge_scorer as rs_module
from bert_score import score as bert_score_fn
_rouge = rs_module.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

REFERENCE_REPORTS = {
    'Benign: Fibroadenoma'       : 'Fibroadenoma is a benign breast tumor with biphasic proliferation of epithelial and stromal tissue. No malignancy. Danger LOW. Urgency ROUTINE. No chemotherapy needed. Monitor annually.',
    'Malignant: Ductal Carcinoma': 'Invasive ductal carcinoma is the most common breast cancer arising from milk ducts. Danger HIGH but treatable. Urgency URGENT. May need surgery, chemotherapy, radiation, hormone or targeted therapy depending on receptor status.',
    'Benign: Adenosis'           : 'Adenosis is benign lobular proliferation. No malignancy. Danger LOW. Urgency ROUTINE. No chemotherapy. Annual surveillance.',
    'Malignant: Lobular Carcinoma': 'Invasive lobular carcinoma arises from breast lobules. Danger HIGH. Urgency URGENT. Hormone therapy almost always needed. Chemotherapy sometimes needed. MRI essential.',
    'Benign: Phyllodes Tumor'    : 'Benign phyllodes is a fibroepithelial tumour. Danger LOW to MODERATE. Urgency SOON. Surgery with clear margins needed. No chemotherapy.',
    'Malignant: Mucinous Carcinoma': 'Mucinous carcinoma is a special type of breast cancer with mucin pools. Danger MODERATE. Urgency SOON. Favourable prognosis. Hormone therapy first. Chemotherapy often not needed.',
    'Benign: Tubular Adenoma'    : 'Tubular adenoma is a rare benign breast tumour. Danger LOW. Urgency ROUTINE. No treatment needed. Annual imaging.',
    'Malignant: Papillary Carcinoma': 'Papillary carcinoma is a rare breast cancer with favourable prognosis. Danger MODERATE. Urgency SOON. Lumpectomy and hormone therapy. Chemotherapy usually not needed.',
}

def compute_scores(report_text, label):
    ref = REFERENCE_REPORTS.get(label)
    if not ref:
        return None
    rouge  = _rouge.score(ref, report_text)
    P, R, F1 = bert_score_fn([report_text], [ref], lang='en', verbose=False)
    return {
        'ROUGE-1': round(rouge['rouge1'].fmeasure, 4),
        'ROUGE-2': round(rouge['rouge2'].fmeasure, 4),
        'ROUGE-L': round(rouge['rougeL'].fmeasure, 4),
        'BERT-F1': round(float(F1[0]), 4),
    }

def make_scores_chart(scores):
    if not scores:
        return None
    metrics = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BERT-F1']
    vals    = [scores[m] for m in metrics]
    thresholds = [0.35, 0.15, 0.30, 0.84]
    bar_colors = ['#4caf50' if v >= t else '#ff9800' if v >= t*0.6 else '#f44336'
                  for v, t in zip(vals, thresholds)]

    fig, ax = plt.subplots(figsize=(7, 3))
    fig.patch.set_facecolor('#1a1a2e')
    ax.set_facecolor('#16213e')
    bars = ax.bar(metrics, vals, color=bar_colors, edgecolor='#1a1a2e', width=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.4f}', ha='center', fontsize=10,
                fontweight='bold', color='white')
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Score', color='#b0bec5')
    ax.set_title('Report Quality vs Ground Truth', color='white',
                 fontweight='bold', fontsize=11)
    ax.tick_params(colors='white')
    ax.spines[:].set_color('#37474f')
    legend_patches = [
        mpatches.Patch(color='#4caf50', label='Good'),
        mpatches.Patch(color='#ff9800', label='Fair'),
        mpatches.Patch(color='#f44336', label='Low'),
    ]
    ax.legend(handles=legend_patches, loc='upper right',
              facecolor='#1a1a2e', labelcolor='white', fontsize=8)
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=130,
                bbox_inches='tight', facecolor='#1a1a2e')
    plt.close(fig)
    buf.seek(0)
    return PILImage.open(buf)

print('✅ All pipeline functions ready.')

## 5. Gradio UI

In [ ]:
import gradio as gr
import traceback

# ─────────────────────────────────────────────────────────────────────────
# Master pipeline — called on every image submission
# ─────────────────────────────────────────────────────────────────────────
def analyze(image):
    """Full pipeline: image → CNN → chart → LLM report → scores chart."""
    try:
        if image is None:
            return None, '⚠️ Please upload an image first.', None, ''

        # Save uploaded PIL image to temp file for OpenCV
        tmp = '/tmp/uploaded_slide.png'
        if isinstance(image, np.ndarray):
            cv2.imwrite(tmp, cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
        else:
            image.save(tmp)

        # ── Step 1: CNN inference ──────────────────────────────────────
        result = run_cnn(tmp)

        # ── Step 2: Confidence chart ───────────────────────────────────
        conf_chart = make_confidence_chart(result)

        # ── Step 3: LLM report ────────────────────────────────────────
        report_text = generate_report(result)

        # ── Step 4: Score vs ground truth ────────────────────────────
        scores = compute_scores(report_text, result['display_label'])
        scores_chart = make_scores_chart(scores)

        # ── Step 5: Format score summary ─────────────────────────────
        if scores:
            interp = {
                'ROUGE-1': ('Good ✅' if scores['ROUGE-1'] > 0.35 else
                            'Fair ⚠️'  if scores['ROUGE-1'] > 0.20 else 'Low ❌'),
                'ROUGE-2': ('Good ✅' if scores['ROUGE-2'] > 0.15 else
                            'Fair ⚠️'  if scores['ROUGE-2'] > 0.08 else 'Low ❌'),
                'ROUGE-L': ('Good ✅' if scores['ROUGE-L'] > 0.30 else
                            'Fair ⚠️'  if scores['ROUGE-L'] > 0.18 else 'Low ❌'),
                'BERT-F1': ('Strong ✅' if scores['BERT-F1'] > 0.88 else
                            'Moderate ⚠️' if scores['BERT-F1'] > 0.84 else 'Weak ❌'),
            }
            score_summary = (
                f"### 📐 Report Quality vs. Medical Ground Truth\n"
                f"| Metric | Score | Rating |\n"
                f"|--------|-------|--------|\n"
                f"| ROUGE-1 | {scores['ROUGE-1']:.4f} | {interp['ROUGE-1']} |\n"
                f"| ROUGE-2 | {scores['ROUGE-2']:.4f} | {interp['ROUGE-2']} |\n"
                f"| ROUGE-L | {scores['ROUGE-L']:.4f} | {interp['ROUGE-L']} |\n"
                f"| BERT-F1 | {scores['BERT-F1']:.4f} | {interp['BERT-F1']} |\n\n"
                f"**BERT-F1 > 0.84** means the AI report is semantically aligned with "
                f"verified medical literature — not gibberish."
            )
        else:
            score_summary = '_No reference report available for this subtype._'

        return conf_chart, report_text, scores_chart, score_summary

    except Exception as e:
        err = traceback.format_exc()
        return None, f'❌ Error:\n{err}', None, ''


# ─────────────────────────────────────────────────────────────────────────
# Gradio Layout
# ─────────────────────────────────────────────────────────────────────────
DARK_CSS = """
body, .gradio-container { background: #0f0f1a !important; color: #e0e0e0 !important; }
.gr-button-primary { background: #7c4dff !important; border: none; }
.gr-button-primary:hover { background: #9c6dff !important; }
h1, h2, h3 { color: #ce93d8 !important; }
.gr-panel { background: #16213e !important; border: 1px solid #37474f !important; }
.gr-textbox textarea { background: #1a1a2e !important; color: #e0e0e0 !important; 
                        font-family: 'Segoe UI', sans-serif; font-size: 14px; }
.gr-markdown { color: #cfd8dc !important; }
.gr-image { border: 2px solid #37474f !important; border-radius: 8px; }
"""

with gr.Blocks(css=DARK_CSS, title='Breast Cancer AI Advisor') as demo:

    # ── Header ─────────────────────────────────────────────────────────
    gr.Markdown("""
    # 🩺 Breast Cancer AI Advisor
    ### Powered by DenseNet121 (CNN) + Mistral-7B (LLM)

    Upload a breast histopathology image to get:
    - **CNN prediction** with confidence percentages for all subtypes
    - **Full patient-friendly report** — danger level, doctor urgency, treatment, chemo guidance
    - **Quality verification** — ROUGE & BERTScore vs. medical ground truth

    > ⚠️ This tool is for research/educational purposes. Always consult a qualified doctor.
    """)

    # ── Main row ───────────────────────────────────────────────────────
    with gr.Row():

        # Left column — input
        with gr.Column(scale=1):
            gr.Markdown('### 📤 Upload Histopathology Image')
            image_input = gr.Image(
                label='Drop or click to upload (.png / .jpg)',
                type='pil',
                height=300,
            )
            analyze_btn = gr.Button('🔬 Analyze Image', variant='primary', size='lg')
            gr.Markdown("""
            **Supported subtypes:**
            - 🟢 Benign: Adenosis, Fibroadenoma, Phyllodes Tumor, Tubular Adenoma
            - 🔴 Malignant: Ductal, Lobular, Mucinous, Papillary Carcinoma
            """)

        # Right column — confidence chart
        with gr.Column(scale=2):
            gr.Markdown('### 📊 CNN Prediction & Confidence')
            conf_chart_out = gr.Image(
                label='Classification Result',
                type='pil',
                height=300,
            )

    gr.Markdown('---')

    # ── Report row ─────────────────────────────────────────────────────
    gr.Markdown('### 📋 AI Patient Advisory Report')
    report_out = gr.Textbox(
        label='Generated Report (Mistral-7B)',
        lines=28,
        max_lines=60,
        show_copy_button=True,
        placeholder='Your detailed medical advisory report will appear here after analysis...',
    )

    gr.Markdown('---')

    # ── Scores row ─────────────────────────────────────────────────────
    gr.Markdown('### ✅ Report Verification — Is the AI telling the truth?')
    with gr.Row():
        with gr.Column(scale=1):
            scores_chart_out = gr.Image(
                label='ROUGE & BERTScore vs Medical Ground Truth',
                type='pil',
                height=260,
            )
        with gr.Column(scale=1):
            score_summary_out = gr.Markdown(
                value='_Scores will appear here after analysis._'
            )

    gr.Markdown("""
    ---
    **How to read the scores:**
    - **ROUGE** measures word/phrase overlap with verified medical reference text.
    - **BERTScore F1 > 0.84** = the AI report is semantically aligned with real medical literature ✅
    - **BERTScore F1 < 0.80** = the report may contain hallucinations — treat with caution ⚠️
    """)

    # ── Wire up ────────────────────────────────────────────────────────
    analyze_btn.click(
        fn=analyze,
        inputs=[image_input],
        outputs=[conf_chart_out, report_out, scores_chart_out, score_summary_out],
    )


# ── Launch ──────────────────────────────────────────────────────────────
# share=True gives you a public Gradio URL that works from Kaggle
demo.launch(share=True, debug=True)